In [ ]:
import pandas as pd

In [ ]:
df.columns

In [ ]:
df_assessment

# Đây là để thêm cột subject name từ DF_FLM -> enrich context lên tí

In [ ]:
# Giả sử bạn có:


# df_merged = df_assessment.merge(
#     df_flm[['SubjectCode', 'Subject Name']],
#     on='SubjectCode',
#     how='left'
# )


In [ ]:
df_flm=df_flm.drop(columns=['content'])

df_flm

In [ ]:
df_merged.head(1)

In [ ]:
df_flm.columns

# Bỏ khoảng trắng dính chữ

In [ ]:
import re


def fix_spacing_in_text(text):
    if not isinstance(text, str):
        return text

    # Chèn khoảng trắng sau % nếu dính chữ hoa tiếng Việt / Anh
    text = re.sub(r'(%)([A-ZĐÁÀÂĂÉÈÊÍÌÓÒÔƠÚÙƯÝ])', r'\1 \2', text)

    # Chèn khoảng trắng giữa chữ thường/tv và chữ hoa (slotIn → slot In)
    text = re.sub(r'([a-zà-ỹ])([A-Z])', r'\1 \2', text)

    # Chèn khoảng trắng giữa chữ và số (e.g. FE:=>5Final → FE:=>5 Final)
    text = re.sub(r'(\d)([A-Z])', r'\1 \2', text)

    # Chèn khoảng trắng giữa số và chữ thường (e.g. 4points → 4 points)
    text = re.sub(r'(\d)([a-z])', r'\1 \2', text)

    # Chèn khoảng trắng giữa ) và từ tiếp theo nếu bị dính (e.g. điểm)Final → điểm) Final)
    text = re.sub(r'(\))([A-ZÀ-Ỹ])', r'\1 \2', text)

    # Chèn khoảng trắng giữa từ và số (e.g. FinalResult:=>5 → Final Result: =>5)
    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)

    # Normalize nhiều khoảng trắng
    text = re.sub(r'\s{2,}', ' ', text)

    return text.strip()


    
# Cột nào có khả năng dính chữ — áp dụng hàm
columns_to_fix = [ 'Semester', 'NoCredit',
        'Syllabus ID:', 'Degree Level:',
       'Time Allocation:', 'Description:', 'StudentTasks:', 'Tools:',
       'Scoring Scale:', 'DecisionNo MM/dd/yyyy:', 'IsApproved:', 'Note:',
       'MinAvgMarkToPass:', 'IsActive:', 'ApprovedDate:',]

for col in columns_to_fix:
    if col in df_flm.columns:
        df_flm[col] = df_flm[col].apply(fix_spacing_in_text)
print(df_flm.head(10))
# df_flm=df_flm.drop(columns=['content'])


In [ ]:
import json

# Chunk mỗi dòng thành JSON
json_chunks = df_flm.to_dict(orient='records')

# # Lưu ra file nếu cần
# with open("syllabus_chunks.json", "w", encoding="utf-8") as f:
#     json.dump(json_chunks, f, indent=2, ensure_ascii=False)


In [ ]:
json_chunks[9]

In [ ]:
def format_row(row):
    def safe_get(key, default="N/A"):
        val = row[key] if key in row else default
        return str(val).strip() if pd.notnull(val) else default

    def format_block(title, content_key):
        content = safe_get(content_key, "")
        lines = [f"- {line.strip()}" for line in content.splitlines() if line.strip()]
        return f"--- {title} ---\n" + "\n".join(lines) + f"\n--- END {title} ---\n"

    return (
        f"TYPE: overview\n"
        f"Subject Code: {safe_get('SubjectCode')}\n"
        f"Subject Name: {safe_get('Subject Name')}\n"
        f"Degree Level: {safe_get('Degree Level:', 'N/A')} | "
        f"Credits: {safe_get('NoCredit', 'N/A')} | "
        f"Semester: {safe_get('Semester', 'N/A')}\n"
        f"Belong To Combo: {safe_get('BelongToCombo', 'None')}\n"
        f"Pre-requisites: {safe_get('PreRequisite', 'None')}\n"
        f"Scoring Scale: {safe_get('Scoring Scale:', 'N/A')} | "
        f"Min Avg Mark to Pass: {safe_get('MinAvgMarkToPass:', 'N/A')}\n"
        f"Approved: {safe_get('IsApproved:', 'N/A')} on {safe_get('ApprovedDate:', 'N/A')}\n"
        f"Subject Link: {safe_get('SubjectLink')}\n\n"
        f"--- TIME ALLOCATION ---\n{safe_get('Time Allocation:', 'N/A')}\n--- TIME ALLOCATION END ---\n\n"
        + format_block("DESCRIPTION", "Description:")
        + format_block("STUDENT TASKS", "StudentTasks:")
        + format_block("TOOLS", "Tools:")
        + format_block("NOTE", "Note:")
    )



# Tạo list văn bản
text_chunks = [format_row(row) for _, row in df_flm.iterrows()]




In [ ]:
# # # Lưu ra file text nếu cần
# with open("syllabus_chunks.txt", "w", encoding="utf-8") as f:
#     f.write("\n".join(text_chunks))

In [ ]:
for i in text_chunks[20:30]:
    print(i)
    print('-'*60)


# Mấy cái hàm build payload chắc ae kêu chatgpt có cột chi rồi build tương tự hay kiếm cách mô linh hoạt hơn cũng được, NHỚ CÓ MỤC TYPE Ở CONTENT VÀ CHỖ PAYLOAD LUÔN

In [ ]:
def build_payload(row, chunk_type="overview"):
    return {
        "subject_code": str(row.get("SubjectCode", "")).strip(),
        "subject_name": str(row.get("Subject Name", "")).strip(),
        "degree_level": str(row.get("Degree Level:", "")).strip(),
        "semester": int(row.get("Semester", 0)),
        "credits": int(row.get("NoCredit", 0)),
        "belong_to_combo": str(row.get("BelongToCombo", "None")).strip(),
        "type": chunk_type,
        "content": format_row(row),
        "subject_link": str(row.get("SubjectLink", "")).strip()
    }

payloads = [build_payload(row) for _, row in df_flm.iterrows()]


In [ ]:
payloads

In [ ]:
import json

    json.dump(payloads, f, indent=2, ensure_ascii=False)


In [ ]:
def build_session_payload(row):
    def safe(x):
        return str(x).strip() if pd.notnull(x) else "N/A"

    return {
        "subject_code": safe(row["SubjectCode"]),
        "subject_name": safe(row["Subject Name"]),
        "session_no": int(row["Session No"]),
        "lesson_name": safe(row["Name"]),
        "topic": safe(row["Details"]),
        "type": "construtive_question",  # phân loại rõ ràng
        "content": (
            f"TYPE: construtive_question\n"
            f"Subject: {safe(row['SubjectCode'])} - {safe(row['Subject Name'])}\n"
            f"Session: {safe(row['Session No'])} | Lesson: {safe(row['Name'])}\n"
            f"Topic: {safe(row['Details'])}"
        )
    }
session_payloads = [build_session_payload(row) for _, row in df_merged.iterrows()]


In [ ]:
session_payloads[0]['content']

In [ ]:
session_payloads

In [ ]:
import json

with open("cons_questions_payloads.json", "w", encoding="utf-8") as f:
    json.dump(session_payloads, f, indent=2, ensure_ascii=False)

In [ ]:
def build_assessment_payload(row):
    def safe(x):
        return str(x).strip() if pd.notnull(x) else "N/A"

    return {
        "subject_code": safe(row["SubjectCode"]),
        "subject_name": safe(row["Subject Name"]),
        "type": "assessment",
        "category": safe(row["Category"]),
        "part": row.get("Part"),
        "weight": safe(row.get("Weight")),
        "content": (
            f"TYPE: assessment\n"
            f"Subject: {safe(row['SubjectCode'])} - {safe(row['Subject Name'])}\n"
            f"Category: {safe(row['Category'])} | Part: {safe(row.get('Part'))} | Weight: {safe(row.get('Weight'))}\n"
            f"Question Type: {safe(row.get('Question Type'))}\n"
            f"Knowledge and Skill: {safe(row.get('Knowledge and Skill'))}\n"
            f"Grading Guide: {safe(row.get('Grading Guide'))}\n"
            f"Completion Criteria: {safe(row.get('Completion Criteria'))}\n"
            f"Duration: {safe(row.get('Duration'))}\n"
            f"Note: {safe(row.get('Note'))}"
        )
    }
assessment_payloads = [build_assessment_payload(row) for _, row in df_merged.iterrows()]


In [ ]:
assessment_payloads[10]

In [ ]:
import json

with open("assessment_payloads.json", "w", encoding="utf-8") as f:
    json.dump(assessment_payloads, f, indent=2, ensure_ascii=False)
